# Stream E: Depth Pro on C3VD (zero-shot, raw frames)

Model: `apple/DepthPro-hf`, used as released (no training or fine-tuning). Frames go in as stored; only Depth Pro's own preprocessing is applied. Code: `results/E/run_depthpro.py`. Dev = trans_t1_a, test = trans_t2_b. Settings were fixed on dev before the single test run.

**Scaling:** `none` = raw metric output; `dev_scale` = one factor learnt on dev (deployable); `median` / `lsq_depth` = per-frame fits using that frame's GT (oracle, same as the ZoeDepth notebook).

**Masks:** `gt>0` drops only pixels with no GT; `valid` also drops 100 mm clamped and near-black pixels (as in `env_check.py`).

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parents[1] if Path.cwd().name == 'E' else Path.cwd()
SPLIT = 'test'   # 'dev' or 'test'

# Predictions are cached in results/E/cache/, so this is fast after the first run
r = subprocess.run([sys.executable, 'results/E/run_depthpro.py', '--split', SPLIT], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:] if r.returncode == 0 else r.stderr[-3000:])

In [ ]:
summary = pd.read_csv(REPO / 'results' / 'E' / f'depthpro_{SPLIT}_summary.csv')
cols = {'method': 'Model', 'scaling': 'Scaling', 'mae': 'MAE (mm) ↓', 'absrel': 'AbsRel ↓', 'rmse': 'RMSE (mm) ↓', 'delta1': 'δ<1.25 ↑'}
for mk in ['gt>0', 'valid']:
    t = summary[summary['mask'] == mk].drop(columns='mask').rename(columns=cols)
    display(t.style.format({c: '{:.3f}' for c in list(cols.values())[2:]}).hide(axis='index')
            .set_caption(f'Depth Pro vs constant median, {SPLIT}, mask = {mk}'))

In [ ]:
import json
print(json.dumps(json.load(open(REPO / 'results' / 'E' / f'run_info_{SPLIT}.json')), indent=2))